# 🌿 Demeter SDK — Demo en Google Colab

Este cuaderno prueba el SDK de Demeter contra el entorno de staging en `http://localhost:8001`
(o el servidor remoto si está desplegado). Cambia `BASE_URL` y `API_KEY` según tu entorno.

**Pasos:**
1. Instalar el SDK desde Test PyPI
2. Verificar conexión con `ping()`
3. Descargar telemetría real + enriquecer con ciencia agronómica
4. Generar gráficas inline (VPD, timeseries, boxplot)
5. Exportar informe Excel y descargarlo

In [ ]:
# ─── Celda 1: Instalación ─────────────────────────────────────────────────────
# Instalar desde Test PyPI (entorno sandbox)
!pip install --index-url https://test.pypi.org/simple/ \
             --extra-index-url https://pypi.org/simple/ \
             demeter_sdk -q

# Verificar versión
import demeter_sdk
print(f'Demeter SDK v{demeter_sdk.__version__} instalado ✅')

In [ ]:
# ─── Celda 2: Configuración ───────────────────────────────────────────────────
# Ajusta estas variables a tu entorno

# Para staging LOCAL (laptop con compose levantado + ngrok/cloudflare tunnel):
BASE_URL = 'http://tu-ip-o-dominio:8001'   # ← CAMBIAR
API_KEY  = 'tu-api-key-aqui'               # ← CAMBIAR (ver .env → DEMETER_API_KEY)

# Experimento a analizar
EXP_ID = 1
DIAS   = 30
FECHA_SIEMBRA = '2025-09-01'

In [ ]:
# ─── Celda 3: Crear cliente y verificar conexión ──────────────────────────────
from demeter_sdk import DemeterClient

client = DemeterClient(api_key=API_KEY, base_url=BASE_URL)

ok = client.ping()
print(f'Server status: {"✅ Online" if ok else "❌ Offline — revisa BASE_URL y que el servidor esté levantado"}')

if not ok:
    raise RuntimeError('Servidor no accesible. Salta a la Celda 4b para usar datos de demostración offline.')

In [ ]:
# ─── Celda 4a: Descargar y enriquecer datos reales ───────────────────────────
df = client.get_enriched_data(
    experimento_id=EXP_ID,
    dias=DIAS,
    fecha_siembra=FECHA_SIEMBRA,
)

print(f'Registros descargados: {len(df)}')
print(f'Columnas: {list(df.columns)}')
df[['timestamp', 'temperature', 'humidity', 'vpd_kpa', 'dew_point_c', 'dap_days']].head(10)

In [ ]:
# ─── Celda 4b: Datos offline de demostración (sin servidor) ──────────────────
# Ejecuta ESTA CELDA si no tienes servidor disponible
import numpy as np, pandas as pd
from datetime import datetime, timedelta
from demeter_sdk import transform, science

base = datetime(2025, 10, 1)
raw = [{
    'timestamp': (base + timedelta(hours=i)).isoformat(),
    'temperature': round(22 + 6 * np.sin(i / 10), 2),
    'humidity':    round(72 + 12 * np.cos(i / 8), 2),
    'node_id':     (i % 3) + 1,
} for i in range(720)]   # 30 días horarios

df_clean = transform.pipeline(raw, resample_rule='1h')
df = science.enrich(df_clean, fecha_siembra='2025-09-01')

print(f'Demo datos generados: {len(df)} registros')
df[['timestamp', 'temperature', 'vpd_kpa', 'dew_point_c', 'dap_days']].head(5)

In [ ]:
# ─── Celda 5: Estadísticas resumen ───────────────────────────────────────────
cols = ['temperature', 'humidity', 'vpd_kpa', 'dew_point_c', 'wet_bulb_c']
df[cols].describe().round(3)

In [ ]:
# ─── Celda 6: Gráfica VPD con zonas de estrés ────────────────────────────────
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from demeter_sdk import viz
import matplotlib.pyplot as plt

fig = viz.plot_vpd(df)
plt.show()

In [ ]:
# ─── Celda 7: Timeseries de temperatura ──────────────────────────────────────
fig = viz.plot_timeseries(df, sensor='temperature', moving_avg=True)
plt.show()

In [ ]:
# ─── Celda 8: Boxplot por nodo ────────────────────────────────────────────────
fig = viz.plot_boxplot(df, metric='vpd_kpa')
plt.show()

In [ ]:
# ─── Celda 9: Heatmap temperatura × nodo × tiempo ────────────────────────────
fig = viz.plot_heatmap(df, metric='temperature', resample_rule='1D')
plt.show()

In [ ]:
# ─── Celda 10: Exportar informe Excel y descargarlo ──────────────────────────
from demeter_sdk import export

path = export.to_excel(
    df_enriched=df,
    path='/tmp/demeter_report.xlsx',
    experiment_name=f'Experimento {EXP_ID} — Staging Demo',
)
print(f'Excel generado: {path}')

# Descargar en Colab
try:
    from google.colab import files
    files.download(str(path))
    print('Descarga iniciada ✅')
except ImportError:
    print(f'(No estás en Colab) Archivo en: {path}')

In [ ]:
# ─── Celda 11: Exportar CSV ───────────────────────────────────────────────────
csv_path = export.to_csv(df, '/tmp/telemetria_enriquecida.csv')
print(f'CSV: {csv_path} ({csv_path.stat().st_size // 1024} KB)')

try:
    from google.colab import files
    files.download(str(csv_path))
except ImportError:
    pass

---
## ✅ Validación completa

Si todas las celdas anteriores se ejecutaron sin errores:
- **Conexión:** El servidor de staging es accesible desde internet
- **Datos:** El SDK descarga y parsea correctamente la telemetría
- **Ciencia:** Las 15 fórmulas agronómicas funcionan con datos reales
- **Visualización:** Matplotlib genera los 4 tipos de gráficas
- **Export:** Excel y CSV se generan y descargan correctamente

**SDK version:** `demeter_sdk.__version__`